# Agentic AI Notebook

A lightweight agentic application inspired by Claude Code. Given a task, the agent works through three blended phases:

1. **Gather Context** — understand the environment, read files, search for relevant information
2. **Take Action** — execute tools, write/edit files, run commands based on the gathered context
3. **Verify Results** — confirm the actions succeeded and the task is complete

These phases are not rigid stages — they blend together naturally. The agent may gather more context mid-action, or take corrective action during verification.

The system is **expandable** via a plugin architecture for custom **tools**, **skills**, and **MCP servers** — including built-in browser automation powered by Playwright.

## 1. Setup & Configuration

In [ ]:
import os
import json
import subprocess
import textwrap
import base64
import asyncio
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Callable
from enum import Enum

# -- LLM client setup --
# Install: pip install openai playwright nest_asyncio
# Then run: playwright install chromium
# Works with OpenAI, Azure OpenAI, or any OpenAI-compatible API (e.g. Anthropic via proxy, Ollama)
from openai import OpenAI

# Configure your LLM provider here
LLM_CONFIG = {
    "api_key": os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    "base_url": os.environ.get("OPENAI_BASE_URL", None),  # Set for Azure/Ollama/other providers
    "model": os.environ.get("LLM_MODEL", "gpt-4o"),
}

client = OpenAI(
    api_key=LLM_CONFIG["api_key"],
    base_url=LLM_CONFIG["base_url"],
)

WORKING_DIR = Path.cwd()
print(f"Working directory: {WORKING_DIR}")
print(f"LLM model: {LLM_CONFIG['model']}")

## 2. Core Data Structures

The agent's state is tracked through simple dataclasses. Each tool call, its result, and the current phase are recorded so the agent can reason about what has happened and what to do next.

In [ ]:
class Phase(str, Enum):
    """The three agent phases. These blend together during execution."""
    GATHER = "gather_context"
    ACT = "take_action"
    VERIFY = "verify_results"


@dataclass
class ToolResult:
    """Outcome of a single tool invocation."""
    tool_name: str
    arguments: dict
    output: str
    success: bool
    phase: Phase


@dataclass
class AgentState:
    """Tracks the full lifecycle of an agent run."""
    task: str
    phase: Phase = Phase.GATHER
    history: list[ToolResult] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    done: bool = False
    final_answer: str = ""

    @property
    def summary(self) -> str:
        lines = [f"Task: {self.task}", f"Phase: {self.phase.value}", f"Steps taken: {len(self.history)}"]
        for i, result in enumerate(self.history, 1):
            status = "OK" if result.success else "FAIL"
            lines.append(f"  {i}. [{result.phase.value}] {result.tool_name}({result.arguments}) -> {status}")
        return "\n".join(lines)

## 3. Tool Registry — Extensible Plugin System

Tools are registered with a decorator. Each tool provides:
- A **name** used by the LLM to invoke it
- A **description** the LLM reads to decide when to use it
- A **JSON Schema** for its parameters
- A **Python function** that executes the tool

Adding a new tool is as simple as writing a function and decorating it.

In [ ]:
@dataclass
class ToolDef:
    """Definition of a registered tool."""
    name: str
    description: str
    parameters: dict  # JSON Schema
    func: Callable[..., str]
    phase_hint: Phase | None = None  # Optional: suggest which phase this tool belongs to

    def to_openai_tool(self) -> dict:
        """Convert to the OpenAI function-calling format."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }


class ToolRegistry:
    """Central registry for all available tools."""

    def __init__(self):
        self._tools: dict[str, ToolDef] = {}

    def register(
        self,
        name: str,
        description: str,
        parameters: dict,
        phase_hint: Phase | None = None,
    ):
        """Decorator to register a function as a tool."""
        def decorator(func: Callable[..., str]):
            self._tools[name] = ToolDef(
                name=name,
                description=description,
                parameters=parameters,
                func=func,
                phase_hint=phase_hint,
            )
            return func
        return decorator

    def get(self, name: str) -> ToolDef | None:
        return self._tools.get(name)

    def list_tools(self) -> list[str]:
        return list(self._tools.keys())

    def to_openai_tools(self) -> list[dict]:
        return [t.to_openai_tool() for t in self._tools.values()]

    def execute(self, name: str, arguments: dict) -> tuple[str, bool]:
        """Execute a tool by name. Returns (output, success)."""
        tool = self._tools.get(name)
        if not tool:
            return f"Error: Unknown tool '{name}'", False
        try:
            result = tool.func(**arguments)
            return str(result), True
        except Exception as e:
            return f"Error executing {name}: {e}", False


# Global registry — tools register themselves here
registry = ToolRegistry()
print(f"Tool registry initialized.")

## 4. Built-in Tools

A starter set of tools that cover the basics: reading files, listing directories, searching content, running shell commands, and writing files. These mirror the core capabilities of Claude Code.

In [ ]:
# ---------- Gather-phase tools ----------

@registry.register(
    name="read_file",
    description="Read the contents of a file. Returns the file content as a string.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Path to the file (relative to working dir)"},
        },
        "required": ["path"],
    },
    phase_hint=Phase.GATHER,
)
def read_file(path: str) -> str:
    target = WORKING_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"File not found: {target}")
    content = target.read_text(encoding="utf-8", errors="replace")
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + f"\n... [truncated, {len(content)} chars total]"
    return content


@registry.register(
    name="list_directory",
    description="List files and directories at a given path. Returns names with [dir] or [file] markers.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Directory path (relative to working dir). Defaults to '.'"},
        },
        "required": [],
    },
    phase_hint=Phase.GATHER,
)
def list_directory(path: str = ".") -> str:
    target = WORKING_DIR / path
    if not target.is_dir():
        raise NotADirectoryError(f"Not a directory: {target}")
    entries = sorted(target.iterdir())
    lines = []
    for entry in entries:
        if entry.name.startswith("."):
            continue
        kind = "[dir] " if entry.is_dir() else "[file]"
        lines.append(f"{kind} {entry.name}")
    return "\n".join(lines) if lines else "(empty directory)"


@registry.register(
    name="search_files",
    description="Search for a text pattern across files. Returns matching lines with file paths and line numbers.",
    parameters={
        "type": "object",
        "properties": {
            "pattern": {"type": "string", "description": "Text or regex pattern to search for"},
            "glob": {"type": "string", "description": "File glob to limit search (e.g. '*.py'). Default: '*'"},
        },
        "required": ["pattern"],
    },
    phase_hint=Phase.GATHER,
)
def search_files(pattern: str, glob: str = "*") -> str:
    import re
    results = []
    for filepath in WORKING_DIR.rglob(glob):
        if not filepath.is_file() or ".git" in filepath.parts:
            continue
        try:
            text = filepath.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        for line_num, line in enumerate(text.splitlines(), 1):
            if re.search(pattern, line):
                rel = filepath.relative_to(WORKING_DIR)
                results.append(f"{rel}:{line_num}: {line.rstrip()}")
        if len(results) >= 50:
            results.append("... (results truncated at 50 matches)")
            break
    return "\n".join(results) if results else f"No matches found for pattern: {pattern}"


# ---------- Action-phase tools ----------

@registry.register(
    name="write_file",
    description="Write content to a file. Creates the file if it doesn't exist, overwrites if it does.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path (relative to working dir)"},
            "content": {"type": "string", "description": "Content to write"},
        },
        "required": ["path", "content"],
    },
    phase_hint=Phase.ACT,
)
def write_file(path: str, content: str) -> str:
    target = WORKING_DIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Wrote {len(content)} chars to {path}"


@registry.register(
    name="edit_file",
    description="Replace a specific string in a file with new content. The old_string must appear exactly once in the file.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path (relative to working dir)"},
            "old_string": {"type": "string", "description": "Exact text to find and replace"},
            "new_string": {"type": "string", "description": "Replacement text"},
        },
        "required": ["path", "old_string", "new_string"],
    },
    phase_hint=Phase.ACT,
)
def edit_file(path: str, old_string: str, new_string: str) -> str:
    target = WORKING_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"File not found: {target}")
    content = target.read_text(encoding="utf-8")
    count = content.count(old_string)
    if count == 0:
        raise ValueError(f"old_string not found in {path}")
    if count > 1:
        raise ValueError(f"old_string appears {count} times in {path} — must be unique")
    new_content = content.replace(old_string, new_string, 1)
    target.write_text(new_content, encoding="utf-8")
    return f"Edited {path}: replaced 1 occurrence"


@registry.register(
    name="run_command",
    description="Run a shell command and return its stdout and stderr. Use for builds, tests, git, etc.",
    parameters={
        "type": "object",
        "properties": {
            "command": {"type": "string", "description": "Shell command to execute"},
        },
        "required": ["command"],
    },
    phase_hint=Phase.ACT,
)
def run_command(command: str) -> str:
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True,
        cwd=str(WORKING_DIR), timeout=60,
    )
    output_parts = []
    if result.stdout.strip():
        output_parts.append(f"STDOUT:\n{result.stdout.strip()}")
    if result.stderr.strip():
        output_parts.append(f"STDERR:\n{result.stderr.strip()}")
    output_parts.append(f"EXIT CODE: {result.returncode}")
    return "\n".join(output_parts)


# ---------- Verify-phase tools ----------

@registry.register(
    name="task_complete",
    description="Signal that the task is finished. Provide a summary of what was done and the outcome.",
    parameters={
        "type": "object",
        "properties": {
            "summary": {"type": "string", "description": "Summary of what was accomplished"},
        },
        "required": ["summary"],
    },
    phase_hint=Phase.VERIFY,
)
def task_complete(summary: str) -> str:
    return f"TASK COMPLETE: {summary}"


print(f"Registered {len(registry.list_tools())} built-in tools: {', '.join(registry.list_tools())}")

## 5. Skills — Higher-Level Composable Actions

Skills are higher-level than tools. A skill is a **multi-step recipe** that composes multiple tool calls to accomplish a common task pattern. Skills are registered separately and the agent can invoke them just like tools.

Think of tools as atoms and skills as molecules.

In [ ]:
@dataclass
class SkillDef:
    """A skill is a multi-step recipe that composes tools."""
    name: str
    description: str
    parameters: dict
    func: Callable[..., str]  # Receives (registry, **kwargs) so it can call tools


class SkillRegistry:
    """Registry for skills. Skills are exposed as tools to the LLM."""

    def __init__(self, tool_registry: ToolRegistry):
        self._skills: dict[str, SkillDef] = {}
        self._tool_registry = tool_registry

    def register(self, name: str, description: str, parameters: dict):
        """Decorator to register a skill."""
        def decorator(func: Callable):
            self._skills[name] = SkillDef(
                name=name,
                description=description,
                parameters=parameters,
                func=func,
            )
            # Also register as a tool so the LLM can call it
            self._tool_registry._tools[name] = ToolDef(
                name=name,
                description=f"[SKILL] {description}",
                parameters=parameters,
                func=lambda **kwargs: func(self._tool_registry, **kwargs),
            )
            return func
        return decorator

    def list_skills(self) -> list[str]:
        return list(self._skills.keys())


skills = SkillRegistry(registry)


# ---------- Built-in Skills ----------

@skills.register(
    name="summarize_project",
    description="Analyze the project structure and produce a summary: directory tree, key files, languages used, and purpose.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Root path to analyze. Defaults to '.'"},
        },
        "required": [],
    },
)
def summarize_project(tool_reg: ToolRegistry, path: str = ".") -> str:
    listing, _ = tool_reg.execute("list_directory", {"path": path})
    key_files = []
    for name in ["README.md", "pyproject.toml", "setup.py", "package.json", "Cargo.toml", "go.mod", "Makefile"]:
        full = Path(WORKING_DIR) / path / name
        if full.is_file():
            key_files.append(name)
    ext_counts: dict[str, int] = {}
    for fp in (WORKING_DIR / path).rglob("*"):
        if fp.is_file() and ".git" not in fp.parts:
            ext = fp.suffix or "(no extension)"
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
    ext_summary = ", ".join(f"{ext}: {c}" for ext, c in sorted(ext_counts.items(), key=lambda x: -x[1])[:10])
    return (
        f"Directory listing:\n{listing}\n\n"
        f"Key project files found: {', '.join(key_files) if key_files else 'none'}\n"
        f"File types: {ext_summary}\n"
        f"Total files: {sum(ext_counts.values())}"
    )


@skills.register(
    name="find_and_replace",
    description="Search for a pattern across the project and replace all occurrences. Returns a report of changes made.",
    parameters={
        "type": "object",
        "properties": {
            "search_pattern": {"type": "string", "description": "Text to search for"},
            "replacement": {"type": "string", "description": "Text to replace with"},
            "glob": {"type": "string", "description": "File glob to limit scope (e.g. '*.py')"},
        },
        "required": ["search_pattern", "replacement"],
    },
)
def find_and_replace(tool_reg: ToolRegistry, search_pattern: str, replacement: str, glob: str = "*") -> str:
    changes = []
    for filepath in WORKING_DIR.rglob(glob):
        if not filepath.is_file() or ".git" in filepath.parts:
            continue
        try:
            content = filepath.read_text(encoding="utf-8")
        except Exception:
            continue
        if search_pattern in content:
            count = content.count(search_pattern)
            new_content = content.replace(search_pattern, replacement)
            filepath.write_text(new_content, encoding="utf-8")
            rel = filepath.relative_to(WORKING_DIR)
            changes.append(f"{rel}: {count} replacement(s)")
    if not changes:
        return f"No occurrences of '{search_pattern}' found."
    return f"Replaced '{search_pattern}' with '{replacement}' in {len(changes)} file(s):\n" + "\n".join(changes)


print(f"Registered {len(skills.list_skills())} built-in skills: {', '.join(skills.list_skills())}")
print(f"Total available tools (tools + skills): {len(registry.list_tools())}")

## 6. MCP Server Framework

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/) servers expose tools over a standardized JSON-RPC transport. Our `MCPServerManager` can connect to external MCP servers and automatically register their tools into our tool registry — making them available to the agent just like built-in tools.

This enables powerful integrations: browser automation, database access, API gateways, and more — all pluggable via config.

In [ ]:
@dataclass
class MCPServer:
    """Represents a connected MCP server."""
    name: str
    command: str
    args: list[str] = field(default_factory=list)
    env: dict[str, str] = field(default_factory=dict)
    process: Any = None
    tools: list[str] = field(default_factory=list)


class MCPServerManager:
    """
    Manages MCP server lifecycle and bridges their tools into the ToolRegistry.

    MCP servers are external processes that expose tools over stdin/stdout JSON-RPC.
    This manager starts them, discovers their tools, and registers proxy functions
    so the agent can call them transparently.
    """

    def __init__(self, tool_registry: ToolRegistry):
        self._servers: dict[str, MCPServer] = {}
        self._registry = tool_registry
        self._request_id = 0

    def _next_id(self) -> int:
        self._request_id += 1
        return self._request_id

    async def connect(self, name: str, command: str, args: list[str] = None, env: dict[str, str] = None):
        """Start an MCP server process and discover its tools."""
        args = args or []
        env = env or {}
        full_env = {**os.environ, **env}

        process = await asyncio.create_subprocess_exec(
            command, *args,
            stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            env=full_env,
        )

        server = MCPServer(name=name, command=command, args=args, env=env, process=process)
        self._servers[name] = server

        # Initialize the MCP session
        init_request = {
            "jsonrpc": "2.0",
            "id": self._next_id(),
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "agentic-notebook", "version": "1.0.0"},
            },
        }
        await self._send(server, init_request)

        # Send initialized notification
        await self._send_notification(server, "notifications/initialized", {})

        # Discover tools
        list_request = {
            "jsonrpc": "2.0",
            "id": self._next_id(),
            "method": "tools/list",
            "params": {},
        }
        response = await self._send(server, list_request)

        if response and "result" in response:
            tools = response["result"].get("tools", [])
            for tool in tools:
                self._register_mcp_tool(server, tool)
            server.tools = [t["name"] for t in tools]
            print(f"  MCP '{name}': registered {len(tools)} tools: {', '.join(server.tools)}")

        return server

    async def _send(self, server: MCPServer, message: dict) -> dict | None:
        """Send a JSON-RPC message and read the response."""
        data = json.dumps(message) + "\n"
        server.process.stdin.write(data.encode())
        await server.process.stdin.drain()
        line = await asyncio.wait_for(server.process.stdout.readline(), timeout=30)
        if line:
            return json.loads(line.decode())
        return None

    async def _send_notification(self, server: MCPServer, method: str, params: dict):
        """Send a JSON-RPC notification (no response expected)."""
        message = {"jsonrpc": "2.0", "method": method, "params": params}
        data = json.dumps(message) + "\n"
        server.process.stdin.write(data.encode())
        await server.process.stdin.drain()

    def _register_mcp_tool(self, server: MCPServer, tool_spec: dict):
        """Register an MCP-discovered tool into our ToolRegistry."""
        tool_name = f"mcp_{server.name}_{tool_spec['name']}"
        description = tool_spec.get("description", f"MCP tool from {server.name}")
        input_schema = tool_spec.get("inputSchema", {"type": "object", "properties": {}})

        captured_server = server
        captured_tool_name = tool_spec["name"]
        captured_manager = self

        def mcp_proxy(**kwargs) -> str:
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            return loop.run_until_complete(
                captured_manager._call_tool(captured_server, captured_tool_name, kwargs)
            )

        self._registry._tools[tool_name] = ToolDef(
            name=tool_name,
            description=f"[MCP:{server.name}] {description}",
            parameters=input_schema,
            func=mcp_proxy,
        )

    async def _call_tool(self, server: MCPServer, tool_name: str, arguments: dict) -> str:
        """Call a tool on an MCP server."""
        request = {
            "jsonrpc": "2.0",
            "id": self._next_id(),
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        }
        response = await self._send(server, request)
        if response and "result" in response:
            content = response["result"].get("content", [])
            parts = []
            for item in content:
                if item.get("type") == "text":
                    parts.append(item["text"])
                elif item.get("type") == "image":
                    parts.append(f"[Image: {item.get('mimeType', 'image/png')}]")
                else:
                    parts.append(json.dumps(item))
            return "\n".join(parts) if parts else "OK"
        elif response and "error" in response:
            return f"MCP Error: {response['error'].get('message', 'Unknown error')}"
        return "No response from MCP server"

    async def disconnect_all(self):
        """Shut down all MCP server processes."""
        for name, server in self._servers.items():
            if server.process:
                server.process.terminate()
                await server.process.wait()
        self._servers.clear()

    def list_servers(self) -> dict[str, list[str]]:
        """Return {server_name: [tool_names]} for all connected servers."""
        return {name: server.tools for name, server in self._servers.items()}


mcp_manager = MCPServerManager(registry)

# ===== Connect an MCP server (example) =====
# Uncomment to connect to an MCP server at startup:
#
# await mcp_manager.connect(
#     name="filesystem",
#     command="npx",
#     args=["-y", "@modelcontextprotocol/server-filesystem", "/path/to/allowed/dir"],
# )

print("MCP Server manager initialized.")

## 7. Browser Automation Tools (Playwright)

A full set of browser tools powered by [Playwright](https://playwright.dev/python/). These let the agent navigate websites, interact with elements, fill forms, take screenshots, and extract content — enabling tasks like "go to Amazon and add items to cart."

The `BrowserSession` manages a singleton Chromium instance that is lazily launched on first use and reused across all tool calls within a session.

**Setup:** `pip install playwright nest_asyncio && playwright install chromium`

In [ ]:
from playwright.sync_api import sync_playwright, Browser, Page, BrowserContext


class BrowserSession:
    """
    Manages a persistent browser instance for the agent.
    Lazily starts Chromium on first access. Reuses the same page across tool calls
    so the agent can navigate, interact, and verify across multiple steps.
    """

    def __init__(self, headless: bool = True):
        self._headless = headless
        self._playwright = None
        self._browser: Browser | None = None
        self._context: BrowserContext | None = None
        self._page: Page | None = None
        self._screenshot_dir = WORKING_DIR / ".browser_screenshots"

    @property
    def page(self) -> Page:
        """Lazily initialize browser and return the active page."""
        if self._page is None or self._page.is_closed():
            self._start()
        return self._page

    def _start(self):
        self._playwright = sync_playwright().start()
        self._browser = self._playwright.chromium.launch(headless=self._headless)
        self._context = self._browser.new_context(
            viewport={"width": 1280, "height": 720},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            ),
        )
        self._page = self._context.new_page()
        self._screenshot_dir.mkdir(exist_ok=True)

    def close(self):
        if self._browser:
            self._browser.close()
        if self._playwright:
            self._playwright.stop()
        self._page = None
        self._browser = None
        self._playwright = None

    def take_screenshot(self, name: str = "screenshot") -> str:
        """Save a screenshot and return the file path."""
        path = self._screenshot_dir / f"{name}.png"
        self.page.screenshot(path=str(path))
        return str(path)


# Global browser session
browser_session = BrowserSession(headless=True)  # Set headless=False to watch the browser


# ---------- Browser Navigation Tools ----------

@registry.register(
    name="browser_navigate",
    description="Navigate the browser to a URL. Returns the page title and current URL after loading.",
    parameters={
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "The URL to navigate to"},
        },
        "required": ["url"],
    },
    phase_hint=Phase.ACT,
)
def browser_navigate(url: str) -> str:
    page = browser_session.page
    page.goto(url, wait_until="domcontentloaded", timeout=30000)
    page.wait_for_load_state("networkidle", timeout=15000)
    return f"Navigated to: {page.url}\nTitle: {page.title()}"


@registry.register(
    name="browser_click",
    description="Click on an element matching a CSS selector or text content. Use CSS selectors like '#id', '.class', 'button', or Playwright text selectors like 'text=Add to Cart'.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector or Playwright selector (e.g. 'text=Submit', '#buy-now', '.add-to-cart')"},
        },
        "required": ["selector"],
    },
    phase_hint=Phase.ACT,
)
def browser_click(selector: str) -> str:
    page = browser_session.page
    page.click(selector, timeout=10000)
    page.wait_for_load_state("networkidle", timeout=10000)
    return f"Clicked: {selector}\nCurrent URL: {page.url}\nTitle: {page.title()}"


@registry.register(
    name="browser_type",
    description="Type text into an input field matching the selector. Clears existing content first.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector for the input field (e.g. '#search', 'input[name=q]', '[placeholder=Search]')"},
            "text": {"type": "string", "description": "Text to type into the field"},
            "press_enter": {"type": "boolean", "description": "Press Enter after typing. Default: false"},
        },
        "required": ["selector", "text"],
    },
    phase_hint=Phase.ACT,
)
def browser_type(selector: str, text: str, press_enter: bool = False) -> str:
    page = browser_session.page
    page.fill(selector, text, timeout=10000)
    if press_enter:
        page.press(selector, "Enter")
        page.wait_for_load_state("networkidle", timeout=10000)
    return f"Typed '{text}' into {selector}" + (" and pressed Enter" if press_enter else "")


@registry.register(
    name="browser_screenshot",
    description="Take a screenshot of the current page. Returns the file path of the saved screenshot.",
    parameters={
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "Name for the screenshot file (without extension). Default: 'screenshot'"},
        },
        "required": [],
    },
    phase_hint=Phase.VERIFY,
)
def browser_screenshot(name: str = "screenshot") -> str:
    path = browser_session.take_screenshot(name)
    return f"Screenshot saved to: {path}"


@registry.register(
    name="browser_get_text",
    description="Extract visible text content from the current page or a specific element. Useful for reading search results, product details, prices, confirmation messages, etc.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector to extract text from. If omitted, returns text from the whole page (truncated to 5000 chars)."},
        },
        "required": [],
    },
    phase_hint=Phase.GATHER,
)
def browser_get_text(selector: str = "body") -> str:
    page = browser_session.page
    element = page.query_selector(selector)
    if not element:
        return f"No element found matching: {selector}"
    text = element.inner_text()
    max_chars = 5000
    if len(text) > max_chars:
        text = text[:max_chars] + f"\n... [truncated, {len(text)} chars total]"
    return text


@registry.register(
    name="browser_get_html",
    description="Get the outer HTML of an element. Useful for inspecting page structure, finding selectors, and understanding element hierarchy.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector for the element. Default: 'body'"},
            "outer": {"type": "boolean", "description": "If true, returns outerHTML; if false, innerHTML. Default: true"},
        },
        "required": [],
    },
    phase_hint=Phase.GATHER,
)
def browser_get_html(selector: str = "body", outer: bool = True) -> str:
    page = browser_session.page
    element = page.query_selector(selector)
    if not element:
        return f"No element found matching: {selector}"
    html = element.evaluate("(el, outer) => outer ? el.outerHTML : el.innerHTML", outer)
    max_chars = 10_000
    if len(html) > max_chars:
        html = html[:max_chars] + f"\n... [truncated, {len(html)} chars total]"
    return html


@registry.register(
    name="browser_select",
    description="Select an option from a dropdown (<select>) element by its value or visible text.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector for the <select> element"},
            "value": {"type": "string", "description": "Option value or label to select"},
        },
        "required": ["selector", "value"],
    },
    phase_hint=Phase.ACT,
)
def browser_select(selector: str, value: str) -> str:
    page = browser_session.page
    page.select_option(selector, value, timeout=10000)
    return f"Selected '{value}' in {selector}"


@registry.register(
    name="browser_wait",
    description="Wait for an element to appear on the page. Useful after navigation or dynamic content loading.",
    parameters={
        "type": "object",
        "properties": {
            "selector": {"type": "string", "description": "CSS selector to wait for"},
            "timeout": {"type": "integer", "description": "Max wait time in milliseconds. Default: 10000"},
        },
        "required": ["selector"],
    },
    phase_hint=Phase.GATHER,
)
def browser_wait(selector: str, timeout: int = 10000) -> str:
    page = browser_session.page
    page.wait_for_selector(selector, timeout=timeout)
    return f"Element '{selector}' is now visible"


@registry.register(
    name="browser_eval_js",
    description="Execute JavaScript in the browser and return the result. Useful for complex interactions, scrolling, or extracting structured data.",
    parameters={
        "type": "object",
        "properties": {
            "script": {"type": "string", "description": "JavaScript code to execute. Use 'return' for expressions."},
        },
        "required": ["script"],
    },
    phase_hint=Phase.ACT,
)
def browser_eval_js(script: str) -> str:
    page = browser_session.page
    result = page.evaluate(script)
    return json.dumps(result, indent=2, default=str) if result is not None else "undefined"


@registry.register(
    name="browser_close",
    description="Close the browser session. Use when done with all browser tasks.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
    },
    phase_hint=Phase.VERIFY,
)
def browser_close() -> str:
    browser_session.close()
    return "Browser session closed."


print(f"Browser tools registered. Total tools: {len(registry.list_tools())}")

## 8. Browser Skills — Shopping, Scraping & More

Higher-level skills that compose browser tools for common workflows. These demonstrate how to build complex multi-step browser automations.

In [ ]:
@skills.register(
    name="web_search",
    description="Search the web using Google and return the top results with titles and URLs.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
            "num_results": {"type": "integer", "description": "Number of results to return. Default: 5"},
        },
        "required": ["query"],
    },
)
def web_search(tool_reg: ToolRegistry, query: str, num_results: int = 5) -> str:
    # Navigate to Google
    tool_reg.execute("browser_navigate", {"url": f"https://www.google.com/search?q={query}"})
    # Extract search results
    text, _ = tool_reg.execute("browser_get_text", {"selector": "#search"})
    return f"Search results for '{query}':\n\n{text}"


@skills.register(
    name="amazon_add_to_cart",
    description="Search for a product on Amazon, select the first result, and add it to cart. Returns confirmation of what was added.",
    parameters={
        "type": "object",
        "properties": {
            "product": {"type": "string", "description": "Product name or search terms (e.g. 'wireless mouse', 'USB-C cable')"},
        },
        "required": ["product"],
    },
)
def amazon_add_to_cart(tool_reg: ToolRegistry, product: str) -> str:
    steps = []

    # Step 1: Navigate to Amazon
    result, ok = tool_reg.execute("browser_navigate", {"url": "https://www.amazon.com"})
    steps.append(f"1. Navigated to Amazon: {result}")

    # Step 2: Search for the product
    result, ok = tool_reg.execute("browser_type", {
        "selector": "#twotabsearchtextbox",
        "text": product,
        "press_enter": True,
    })
    steps.append(f"2. Searched for '{product}'")

    # Step 3: Wait for results and get the first product link
    tool_reg.execute("browser_wait", {"selector": "[data-component-type='s-search-result']"})

    # Step 4: Click the first product result
    result, ok = tool_reg.execute("browser_click", {
        "selector": "[data-component-type='s-search-result'] h2 a",
    })
    steps.append(f"3. Clicked first result: {result}")

    # Step 5: Get the product title and price
    title, _ = tool_reg.execute("browser_get_text", {"selector": "#productTitle"})
    price, _ = tool_reg.execute("browser_get_text", {"selector": ".a-price .a-offscreen"})
    steps.append(f"4. Product: {title.strip()}")
    steps.append(f"   Price: {price.strip()}")

    # Step 6: Click Add to Cart
    result, ok = tool_reg.execute("browser_click", {"selector": "#add-to-cart-button"})
    steps.append(f"5. Clicked 'Add to Cart'")

    # Step 7: Verify — take a screenshot
    screenshot, _ = tool_reg.execute("browser_screenshot", {"name": f"amazon_cart_{product.replace(' ', '_')}"})
    steps.append(f"6. Screenshot saved: {screenshot}")

    return "\n".join(steps)


@skills.register(
    name="scrape_page",
    description="Navigate to a URL, extract the page title, main text content, and all links. Returns structured data.",
    parameters={
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "URL to scrape"},
        },
        "required": ["url"],
    },
)
def scrape_page(tool_reg: ToolRegistry, url: str) -> str:
    # Navigate
    tool_reg.execute("browser_navigate", {"url": url})

    # Get page info
    page = browser_session.page
    title = page.title()
    current_url = page.url

    # Extract main text
    text, _ = tool_reg.execute("browser_get_text", {"selector": "body"})

    # Extract links via JS
    links_json, _ = tool_reg.execute("browser_eval_js", {
        "script": "Array.from(document.querySelectorAll('a[href]')).slice(0, 20).map(a => ({text: a.innerText.trim().substring(0, 80), href: a.href}))"
    })

    return (
        f"Title: {title}\n"
        f"URL: {current_url}\n"
        f"\n--- Page Content ---\n{text}\n"
        f"\n--- Links (top 20) ---\n{links_json}"
    )


print(f"Browser skills registered. Total tools: {len(registry.list_tools())}")
print(f"All skills: {', '.join(skills.list_skills())}")

## 9. The Agent Loop

The agent loop sends the task to the LLM, processes tool calls, feeds results back, and repeats until the LLM signals completion. The three phases blend naturally — the system prompt guides the LLM through them, but it can move freely between phases as needed.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are an autonomous agent that completes tasks by using tools.
    You work through three blended phases:

    1. GATHER CONTEXT — Read files, list directories, search code, browse web pages to understand what you're working with.
    2. TAKE ACTION — Write files, edit code, run commands, interact with the browser to make the required changes.
    3. VERIFY RESULTS — Re-read files, run tests, take screenshots, or check output to confirm your work is correct.

    These phases are not rigid. You may gather more context while acting, or take corrective
    action during verification. Move fluidly between phases as the task requires.

    Browser capabilities:
    - You can navigate to any website using browser_navigate.
    - You can interact with pages: click buttons, fill forms, select dropdowns.
    - You can extract text and HTML from pages to understand content.
    - You can take screenshots to verify visual state.
    - You can execute JavaScript for advanced interactions.
    - For shopping tasks (e.g. Amazon), use the amazon_add_to_cart skill or compose browser tools.
    - For web research, use the web_search skill or navigate directly.

    Guidelines:
    - For file tasks: always read a file before editing it.
    - For browser tasks: use browser_get_text or browser_screenshot to verify actions worked.
    - After making changes, verify them (re-read files, take screenshots, etc.).
    - Use the simplest approach that works.
    - When done, call 'task_complete' with a clear summary.
    - If you get stuck, explain what you tried and what blocked you.
""")

MAX_TURNS = 25  # Safety limit to prevent infinite loops


def infer_phase(tool_name: str) -> Phase:
    """Heuristically determine which phase a tool call belongs to."""
    tool_def = registry.get(tool_name)
    if tool_def and tool_def.phase_hint:
        return tool_def.phase_hint
    return Phase.ACT  # Default


def run_agent(task: str, verbose: bool = True) -> AgentState:
    """Run the agent loop to completion."""
    state = AgentState(task=task)
    state.messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    tools_spec = registry.to_openai_tools()

    for turn in range(MAX_TURNS):
        if verbose:
            print(f"\n{'='*60}")
            print(f"Turn {turn + 1} | Phase: {state.phase.value}")
            print(f"{'='*60}")

        # Call the LLM
        response = client.chat.completions.create(
            model=LLM_CONFIG["model"],
            messages=state.messages,
            tools=tools_spec,
            tool_choice="auto",
        )

        message = response.choices[0].message
        state.messages.append(message.model_dump())

        # If the LLM produced text output, display it
        if message.content:
            if verbose:
                print(f"\nAgent: {message.content}")

        # If no tool calls, the agent is done talking
        if not message.tool_calls:
            state.done = True
            state.final_answer = message.content or ""
            break

        # Process each tool call
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            # Track phase transitions
            new_phase = infer_phase(fn_name)
            if new_phase != state.phase:
                state.phase = new_phase
                if verbose:
                    print(f"\n>> Phase transition -> {state.phase.value}")

            if verbose:
                print(f"\n  Tool: {fn_name}")
                print(f"  Args: {json.dumps(fn_args, indent=2)[:200]}")

            # Execute
            output, success = registry.execute(fn_name, fn_args)

            if verbose:
                display = output[:300] + ("..." if len(output) > 300 else "")
                print(f"  Result ({'OK' if success else 'FAIL'}): {display}")

            # Record
            state.history.append(ToolResult(
                tool_name=fn_name,
                arguments=fn_args,
                output=output,
                success=success,
                phase=state.phase,
            ))

            # Check for task_complete
            if fn_name == "task_complete":
                state.done = True
                state.final_answer = fn_args.get("summary", output)

            # Feed result back to LLM
            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

        if state.done:
            break

    if verbose:
        print(f"\n{'='*60}")
        print(f"Agent finished in {len(state.history)} steps.")
        if state.final_answer:
            print(f"\nFinal answer: {state.final_answer}")
        print(f"{'='*60}")

    return state


print("Agent loop ready.")
print(f"Total tools available: {len(registry.list_tools())}")
print(f"Tools: {', '.join(registry.list_tools())}")

## 10. Example — File System Task

A simple task using the file system tools.

In [ ]:
# -- File system task --

task = """
Explore this project directory. List the files, read any interesting ones,
and give me a summary of what this project is about.
"""

state = run_agent(task.strip())

In [ ]:
# -- Review the execution trace --
print(state.summary)

## 11. Example — Browser Task: Amazon Shopping

The agent uses browser tools to navigate Amazon, search for a product, and add it to the cart. This demonstrates how the three phases blend during browser automation:
- **Gather**: read page content, extract product details
- **Act**: navigate, type search queries, click buttons
- **Verify**: take screenshots, confirm cart contents

In [ ]:
# -- Browser task: Add an item to Amazon cart --
# This will launch a headless browser, navigate to Amazon, search for a product, and add it to cart.

task = """
Go to Amazon.com, search for 'mechanical keyboard', click on the first result,
and add it to the cart. Take a screenshot after adding to cart to confirm.
"""

state = run_agent(task.strip())

In [ ]:
# -- Review the browser task trace --
print(state.summary)

# Clean up the browser when done
browser_session.close()

## 12. Extending — Custom Tools & Skills Template

Copy and customize the templates below. Re-run the cell and the agent will pick up the new tools on its next run.

In [ ]:
# ===== YOUR CUSTOM TOOL TEMPLATE =====
#
# @registry.register(
#     name="my_tool_name",
#     description="What this tool does — the LLM reads this to decide when to use it.",
#     parameters={
#         "type": "object",
#         "properties": {
#             "param1": {"type": "string", "description": "Description of param1"},
#             "param2": {"type": "integer", "description": "Description of param2"},
#         },
#         "required": ["param1"],
#     },
#     phase_hint=Phase.ACT,  # or Phase.GATHER, Phase.VERIFY
# )
# def my_tool_name(param1: str, param2: int = 10) -> str:
#     # Your tool logic here
#     return f"Result: {param1}, {param2}"


# ===== YOUR CUSTOM SKILL TEMPLATE =====
#
# @skills.register(
#     name="my_skill_name",
#     description="A multi-step operation that composes existing tools.",
#     parameters={
#         "type": "object",
#         "properties": {
#             "target": {"type": "string", "description": "What to operate on"},
#         },
#         "required": ["target"],
#     },
# )
# def my_skill_name(tool_reg: ToolRegistry, target: str) -> str:
#     # Compose multiple tool calls
#     step1, _ = tool_reg.execute("list_directory", {"path": target})
#     step2, _ = tool_reg.execute("search_files", {"pattern": "TODO", "glob": "*.py"})
#     return f"Directory:\n{step1}\n\nTODOs found:\n{step2}"


# ===== CONNECT AN MCP SERVER =====
#
# To connect a Playwright MCP server instead of using the built-in browser tools:
#   npm install -g @anthropic/mcp-server-playwright
#
# await mcp_manager.connect(
#     name="playwright",
#     command="npx",
#     args=["-y", "@anthropic/mcp-server-playwright"],
# )

print("Templates ready. Uncomment and customize to add your own tools and skills.")